# Cisco CX CDC: Polars (Portable)**Source**: S3 Parquet (10M rows)  **CDC**: Hash-based diff with Polars (Rust engine)  **Target**: Snowflake table  **Runs on**: Snowflake SPCS and EC2/Docker identically

In [1]:
import importlib
required = ['polars', 'pyarrow', 's3fs']
missing = []
for pkg in required:
    try:
        importlib.import_module(pkg.replace('-', '_'))
    except ImportError:
        missing.append(pkg)
if missing:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + missing + ['-q'])
    print(f"Installed: {missing}")
else:
    print("All packages available.")

All packages available.


In [2]:
import os

def get_snowflake_connection():
    """Connect to Snowflake - works in SPCS or on-prem."""
    try:
        from snowflake.snowpark.context import get_active_session
        session = get_active_session()
        print("Connected via Snowflake SPCS")
        return session, "SPCS"
    except:
        pass
    import snowflake.connector
    conn = snowflake.connector.connect(connection_name=os.getenv("SNOWFLAKE_CONNECTION_NAME", "default"))
    print(f"Connected locally to {conn.account}")
    return conn, "LOCAL"

In [3]:
import os, time
import pyarrow.parquet as pq
import pandas as pd
import numpy as np

S3_BUCKET = "cisco-cx-cdc-pilot"
S3_PREV = "s3://cisco-cx-cdc-pilot/cdc_data/prev_snapshot/"
S3_CURR = "s3://cisco-cx-cdc-pilot/cdc_data/curr_snapshot/"

COMPARE_COLS = ['software_version', 'cpu_utilization', 'memory_utilization',
                'critical_bugs_count', 'contract_status', 'ip_address']

def detect_runtime():
    try:
        from snowflake.snowpark.context import get_active_session
        get_active_session()
        return "SPCS"
    except:
        return "LOCAL"

RUNTIME = detect_runtime()
print(f"Runtime: {RUNTIME}")

if RUNTIME == "SPCS":
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
    print("Reading from Snowflake External Stage (@CDC_S3_STAGE)...")
    t0 = time.time()
    prev_pdf = session.sql("""
        SELECT $1:device_id::VARCHAR AS device_id,
               $1:customer_id::VARCHAR AS customer_id,
               $1:software_version::VARCHAR AS software_version,
               $1:cpu_utilization::FLOAT AS cpu_utilization,
               $1:memory_utilization::FLOAT AS memory_utilization,
               $1:critical_bugs_count::INTEGER AS critical_bugs_count,
               $1:contract_status::VARCHAR AS contract_status,
               $1:ip_address::VARCHAR AS ip_address,
               $1:product_family::VARCHAR AS product_family
        FROM @CISCO_CX_PILOT.LANDING_ZONE.CDC_S3_STAGE/prev_snapshot/
    """).to_pandas()
    prev_pdf.columns = [c.lower() for c in prev_pdf.columns]
    print(f"  Read {len(prev_pdf):,} rows (prev) in {time.time()-t0:.1f}s")
    t0 = time.time()
    curr_pdf = session.sql("""
        SELECT $1:device_id::VARCHAR AS device_id,
               $1:customer_id::VARCHAR AS customer_id,
               $1:software_version::VARCHAR AS software_version,
               $1:cpu_utilization::FLOAT AS cpu_utilization,
               $1:memory_utilization::FLOAT AS memory_utilization,
               $1:critical_bugs_count::INTEGER AS critical_bugs_count,
               $1:contract_status::VARCHAR AS contract_status,
               $1:ip_address::VARCHAR AS ip_address,
               $1:product_family::VARCHAR AS product_family
        FROM @CISCO_CX_PILOT.LANDING_ZONE.CDC_S3_STAGE/curr_snapshot/
    """).to_pandas()
    curr_pdf.columns = [c.lower() for c in curr_pdf.columns]
    print(f"  Read {len(curr_pdf):,} rows (curr) in {time.time()-t0:.1f}s")
else:
    import pyarrow.fs as pafs
    print("Reading directly from S3...")
    fs = pafs.S3FileSystem(region='us-east-1')
    t0 = time.time()
    prev_pdf = pq.read_table(f"{S3_BUCKET}/cdc_data/prev_snapshot", filesystem=fs).to_pandas()
    print(f"  Read {len(prev_pdf):,} rows (prev) in {time.time()-t0:.1f}s")
    t1 = time.time()
    curr_pdf = pq.read_table(f"{S3_BUCKET}/cdc_data/curr_snapshot", filesystem=fs).to_pandas()
    print(f"  Read {len(curr_pdf):,} rows (curr) in {time.time()-t1:.1f}s")

print(f"Previous: {len(prev_pdf):,} | Current: {len(curr_pdf):,}")

Runtime: LOCAL
Reading directly from S3...


  Read 10,000,000 rows (prev) in 9.9s


  Read 10,020,000 rows (curr) in 9.5s
Previous: 10,000,000 | Current: 10,020,000


In [4]:
import polars as pl
print(f'Polars {pl.__version__}')

t_start = time.time()

t0 = time.time()
prev_df = pl.from_pandas(prev_pdf)
curr_df = pl.from_pandas(curr_pdf)
t_create = time.time() - t0

t0 = time.time()
hash_expr = pl.concat_str([pl.col(c).cast(pl.Utf8) for c in COMPARE_COLS], separator='|').hash()
prev_df = prev_df.with_columns(hash_expr.alias('_hash'))
curr_df = curr_df.with_columns(hash_expr.alias('_hash'))
t_hash = time.time() - t0

t0 = time.time()
merged = curr_df.select(['device_id', '_hash']).join(
    prev_df.select(['device_id', '_hash']),
    on='device_id', how='full', suffix='_prev'
)
t_merge = time.time() - t0

t0 = time.time()
n_inserts = merged.filter(pl.col('_hash_prev').is_null()).height
n_deletes = merged.filter(pl.col('_hash').is_null()).height
both = merged.filter(pl.col('_hash').is_not_null() & pl.col('_hash_prev').is_not_null())
n_updates = both.filter(pl.col('_hash') != pl.col('_hash_prev')).height
t_classify = time.time() - t0

total_cdc = time.time() - t_start
print(f'\n=== Polars CDC Results ===')
print(f'  DataFrame create: {t_create:.1f}s')
print(f'  Hash compute:     {t_hash:.1f}s')
print(f'  Merge/Join:       {t_merge:.1f}s')
print(f'  Classify:         {t_classify:.1f}s')
print(f'  TOTAL CDC:        {total_cdc:.1f}s')
print(f'  Inserts: {n_inserts:,} | Updates: {n_updates:,} | Deletes: {n_deletes:,}')

Polars 1.41.2



=== Polars CDC Results ===
  DataFrame create: 11.6s
  Hash compute:     5.9s
  Merge/Join:       2.6s
  Classify:         0.0s
  TOTAL CDC:        20.1s
  Inserts: 25,000 | Updates: 18,952 | Deletes: 5,000


In [5]:
t0 = time.time()
conn, runtime = get_snowflake_connection()

insert_keys = merged.filter(pl.col('_hash_prev').is_null()).select('device_id')
update_keys = both.filter(pl.col('_hash') != pl.col('_hash_prev')).select('device_id')
upsert_ids = pl.concat([insert_keys, update_keys])
upsert_df = curr_df.filter(pl.col('device_id').is_in(upsert_ids['device_id']))

write_df = upsert_df.drop('_hash').to_pandas()
write_df.columns = [c.upper() for c in write_df.columns]

if runtime == "LOCAL":
    from snowflake.connector.pandas_tools import write_pandas
    conn.cursor().execute("USE DATABASE CISCO_CX_PILOT")
    conn.cursor().execute("USE SCHEMA PROCESSED")
    conn.cursor().execute("CREATE TABLE IF NOT EXISTS POLARS_CDC_RESULT (DEVICE_ID VARCHAR, CUSTOMER_ID VARCHAR, SOFTWARE_VERSION VARCHAR, CPU_UTILIZATION FLOAT, MEMORY_UTILIZATION FLOAT, CRITICAL_BUGS_COUNT INTEGER, CONTRACT_STATUS VARCHAR, IP_ADDRESS VARCHAR, PRODUCT_FAMILY VARCHAR)")
    conn.cursor().execute("TRUNCATE TABLE POLARS_CDC_RESULT")
    success, nchunks, nrows, _ = write_pandas(conn, write_df, 'POLARS_CDC_RESULT', database='CISCO_CX_PILOT', schema='PROCESSED')
    print(f'Wrote {nrows:,} rows to Snowflake in {time.time()-t0:.1f}s')
else:
    session = conn
    session.sql("USE DATABASE CISCO_CX_PILOT").collect()
    session.sql("USE SCHEMA PROCESSED").collect()
    session.write_pandas(write_df, 'POLARS_CDC_RESULT', database='CISCO_CX_PILOT', schema='PROCESSED', overwrite=True)
    print(f'Wrote {len(write_df):,} rows to Snowflake in {time.time()-t0:.1f}s')

print(f'\n=== TOTAL PIPELINE (Read + CDC + Write): {(time.time()-t0) + total_cdc:.1f}s ===')

Connected locally to FZB62295


/tmp/ipykernel_1142/1875745848.py:7: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  upsert_df = curr_df.filter(pl.col('device_id').is_in(upsert_ids['device_id']))


Wrote 43,952 rows to Snowflake in 14.7s

=== TOTAL PIPELINE (Read + CDC + Write): 34.7s ===
